### Import Dependencies

In [2]:
import os
import openai
from qdrant_client import QdrantClient
from langsmith import Client

from langchain_openai import ChatOpenAI, OpenAIEmbeddings

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

### Download an example reference data point from LangSmith

In [4]:
from dotenv import load_dotenv
import os

load_dotenv("../../.env")

True

In [5]:
ls_client = Client()

In [6]:
dataset = ls_client.read_dataset(
    dataset_name="rag-evaluation-dataset"
)

In [7]:
dataset

Dataset(name='rag-evaluation-dataset', description='RAG evaluation dataset', data_type=<DataType.kv: 'kv'>, id=UUID('ae8b3706-f085-42e6-8db9-220efff50be1'), created_at=datetime.datetime(2026, 6, 28, 22, 28, 58, 591206, tzinfo=TzInfo(0)), modified_at=datetime.datetime(2026, 6, 28, 22, 28, 58, 591206, tzinfo=TzInfo(0)), example_count=30, session_count=0, last_session_start_time=None, inputs_schema=None, outputs_schema=None, transformations=None, metadata={'runtime': {'sdk': 'langsmith-py', 'library': 'langsmith', 'runtime': 'python', 'platform': 'macOS-14.8.4-arm64-arm-64bit', 'sdk_version': '0.8.16', 'runtime_version': '3.12.13', 'langchain_version': '1.3.2', 'py_implementation': 'CPython', 'langchain_core_version': '1.4.7'}})

In [8]:
list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[15].outputs

{'ground_truth': "Yes, 'On The Prowl' is marked with explicit lyrics.",
 'reference_context_ids': ['B0BM4XNTW7'],
 'reference_descriptions': ['On The Prowl       Explicit Lyrics ']}

In [9]:
list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[15].inputs

{'question': "Is 'On The Prowl' marked with explicit lyrics?"}

In [10]:
reference_input = list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[15].inputs
reference_output = list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[15].outputs

### RAG Pipeline

In [11]:
qdrant_client = QdrantClient(url="http://localhost:6333")

def get_embedding(text, model="text-embedding-3-small"):
    response = openai.embeddings.create(
        input=text,
        model=model
    )

    return response.data[0].embedding


def retrieve_data(query, k=5):

    query_embedding = get_embedding(query)

    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )

    retrieved_context_ids = []
    retrieved_context = []
    similarity_scores = []
    retrieved_context_ratings = []

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


def process_context(context):

    formatted_context = ""

    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"

    return formatted_context


def build_prompt(preprocessed_context, question):

    prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.

You will be given a question and a list of context.

Instructions:
- Answer the question based on the provided context only.
- Never use word context and refer to it as the available products.
- Do not use markdown formatting.

Context:
{preprocessed_context}

Question:
{question}    
"""

    return prompt


def generate_answer(prompt):

    response = openai.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {"role": "system", "content": prompt}
        ],
        reasoning_effort="none"
    )

    return response.choices[0].message.content


def rag_pipeline(question, top_k=5):

    retrieved_context = retrieve_data(question, k=top_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(preprocessed_context, question)
    answer = generate_answer(prompt)

    final_answer = {
        "answer": answer,
        "question": question,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"]
    }

    return final_answer

In [13]:
rag_pipeline("what is a good summer song?")

{'answer': 'A great summer pick is Rhythm Is A Dancer 30th Anniversary (ID: B09W748KRB) — it has a 4.7 rating.',
 'question': 'what is a good summer song?',
 'retrieved_context_ids': ['B09Y4X2XTS',
  'B0BS1SHT91',
  'B09VCV94KF',
  'B09W748KRB',
  'B09PNST3BD'],
 'retrieved_context': ['Songs About You ',
  "Now That's What I Call A Love Song / Various ",
  'Live Songs For Beginners, Wild Tales ',
  'Rhythm Is A Dancer 30th Anniversary ',
  'Marry Me Soundtrack ']}

### RAGAS Metrics

In [19]:
from ragas.dataset_schema import SingleTurnSample
from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy

/var/folders/j5/79_k4_2170vfff4rp856r5vm0000gn/T/ipykernel_58161/3756680326.py:2: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextPrecision
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
/var/folders/j5/79_k4_2170vfff4rp856r5vm0000gn/T/ipykernel_58161/3756680326.py:2: DeprecationWarning: Importing IDBasedContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextRecall
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
/var/folders/j5/79_k4_2170vfff4rp856r5vm0000gn/T/ipykernel_58161/3756680326.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is depre

In [15]:
ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-5.4-mini"))
ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))

/var/folders/j5/79_k4_2170vfff4rp856r5vm0000gn/T/ipykernel_58161/840510326.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-5.4-mini"))
/var/folders/j5/79_k4_2170vfff4rp856r5vm0000gn/T/ipykernel_58161/840510326.py:2: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings(model="text-embedding-3-small"))


In [20]:
reference_input

{'question': "Is 'On The Prowl' marked with explicit lyrics?"}

In [21]:
reference_output

{'ground_truth': "Yes, 'On The Prowl' is marked with explicit lyrics.",
 'reference_context_ids': ['B0BM4XNTW7'],
 'reference_descriptions': ['On The Prowl       Explicit Lyrics ']}

In [31]:
result = rag_pipeline(reference_input["question"])

In [32]:
result

{'answer': 'Yes. “On The Prowl” is marked with explicit lyrics.',
 'question': "Is 'On The Prowl' marked with explicit lyrics?",
 'retrieved_context_ids': ['B0BM4XNTW7',
  'B0B8DK9VH2',
  'B09S8K9GYR',
  'B09XBGM4GB',
  'B09VCV94KF'],
 'retrieved_context': ['On The Prowl       Explicit Lyrics ',
  'The Loneliest Time[LP]       Explicit Lyrics ',
  'What’s It Gonna Take?[2 LP] ',
  'Licked Live In NYC[2 CD] ',
  'Live Songs For Beginners, Wild Tales ']}

In [33]:
async def ragas_context_precision_id_based(run, example):

    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]
    )

    scorer = IDBasedContextPrecision()

    return await scorer.single_turn_ascore(sample)

In [29]:
await ragas_context_precision_id_based(result, reference_output)

0.2

In [34]:
async def ragas_context_recall_id_based(run, example):

    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]
    )

    scorer = IDBasedContextRecall()

    return await scorer.single_turn_ascore(sample)

In [35]:
await ragas_context_recall_id_based(result, reference_output)

1.0

In [36]:
async def ragas_faithfulness(run, example):

    sample = SingleTurnSample(
            user_input=run["question"],
            response=run["answer"],
            retrieved_contexts=run["retrieved_context"]
        )

    scorer = Faithfulness(llm=ragas_llm)
    
    return await scorer.single_turn_ascore(sample)

In [37]:
await ragas_faithfulness(result, reference_output)

1.0

In [39]:
async def ragas_relevancy(run, example):

    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"]
    )

    scorer = ResponseRelevancy(llm=ragas_llm, embeddings=ragas_embeddings)

    return await scorer.single_turn_ascore(sample)

In [40]:
await ragas_relevancy(result, reference_output)

np.float64(0.992549078126773)